# Small-Data Ariel Model Comparison

Goal:
- Train the `ariel_exobiome.ipynb` Ariel quantum regressor on a shared small-data split.
- Train the `ariel_sota.ipynb` FMPE model on the exact same shared train/validation/holdout rows.
- Compare validation and holdout RMSE in one place.

Notes:
- This notebook is designed for a quick local experiment, not a full leaderboard run.
- Defaults are intentionally small so the comparison is practical on CPU.
- The FMPE branch still requires `dingo-gw`, and the ExoBiome branch requires `pennylane`.
- The two models use different feature engineering, but they see the same labeled planets in the shared split.
- Existing cell outputs may still show old `legacy_model` runs until you rerun the notebook from top to bottom.


In [1]:
from __future__ import annotations

import importlib.util
import json
import platform
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return candidate
        nested = candidate / "ariel" / "models" / "notebooks"
        if (nested / "models" / "sbi_ariel_adc2023" / "train.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the Ariel notebook project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT.parent.parent
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "small_data_compare"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
TRAIN_ROWS = 512
VALIDATION_ROWS = 128
HOLDOUT_ROWS = 128

# ExoBiome-style two-stage quantum run.
EXOBIOME_QNN_QUBITS = 12
EXOBIOME_QNN_DEPTH = 2
EXOBIOME_DROPOUT = 0.05
EXOBIOME_WEIGHT_DECAY = 1.0e-4
EXOBIOME_GRADIENT_CLIP = 5.0
EXOBIOME_QNN_INIT_SCALE = 0.1
EXOBIOME_STAGE1_BATCH_SIZE = 64
EXOBIOME_STAGE1_EVAL_BATCH_SIZE = 128
EXOBIOME_STAGE1_MAX_EPOCHS = 18
EXOBIOME_STAGE1_EARLY_STOP = 5
EXOBIOME_STAGE1_SCHEDULER_PATIENCE = 2
EXOBIOME_STAGE1_CLASSICAL_LR = 1.0e-3
EXOBIOME_STAGE2_BATCH_SIZE = 16
EXOBIOME_STAGE2_EVAL_BATCH_SIZE = 32
EXOBIOME_STAGE2_MAX_EPOCHS = 20
EXOBIOME_STAGE2_EARLY_STOP = 6
EXOBIOME_STAGE2_SCHEDULER_PATIENCE = 3
EXOBIOME_STAGE2_CLASSICAL_LR = 5.0e-5
EXOBIOME_STAGE2_QUANTUM_LR = 2.0e-4
EXOBIOME_STAGE2_WARMUP_EPOCHS = 0
EXOBIOME_STAGE2_RAMP_EPOCHS = 10
EXOBIOME_STAGE2_FREEZE_EPOCHS = 4

FMPE_MAX_EPOCHS = 6
FMPE_MAX_STEPS = 60
FMPE_BATCH_SIZE = 64
FMPE_EVAL_BATCH_SIZE = 128
FMPE_POSTERIOR_SAMPLES = 32
FMPE_PERIODIC_SAMPLES = 16

config = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "platform": platform.platform(),
    "train_rows": TRAIN_ROWS,
    "validation_rows": VALIDATION_ROWS,
    "holdout_rows": HOLDOUT_ROWS,
    "exobiome_qubits": EXOBIOME_QNN_QUBITS,
    "exobiome_depth": EXOBIOME_QNN_DEPTH,
    "exobiome_stage1_epochs": EXOBIOME_STAGE1_MAX_EPOCHS,
    "exobiome_stage2_epochs": EXOBIOME_STAGE2_MAX_EPOCHS,
    "fmpe_epochs": FMPE_MAX_EPOCHS,
    "fmpe_max_steps": FMPE_MAX_STEPS,
}
config


{'project_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks',
 'data_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel',
 'output_root': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare',
 'platform': 'macOS-26.2-arm64-arm-64bit',
 'train_rows': 512,
 'validation_rows': 128,
 'holdout_rows': 128,
 'exobiome_qubits': 12,
 'exobiome_depth': 2,
 'exobiome_stage1_epochs': 18,
 'exobiome_stage2_epochs': 20,
 'fmpe_epochs': 6,
 'fmpe_max_steps': 60}

In [2]:
required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "h5py": "h5py",
    "PyYAML": "yaml",
    "PyTorch": "torch",
    "scikit-learn": "sklearn",
    "PennyLane": "pennylane",
    "Dingo": "dingo",
}
missing = [label for label, module_name in required_modules.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Missing Python modules:", ", ".join(missing))
    print()
    print("Suggested local bootstrap commands from", PROJECT_ROOT)
    print("python -m pip install --upgrade pip")
    print("python -m pip install torch")
    print("python -m pip install --no-deps dingo-gw==0.8.3")
    print(f"python -m pip install -r {PROJECT_ROOT / 'models' / 'sbi_ariel_crossgen' / 'requirements-mac.txt'}")
    print("python -m pip install pennylane pennylane-lightning")
    raise RuntimeError("Install the missing dependencies, then rerun this cell.")

import os
import random
import shutil

import h5py
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import yaml
from IPython.display import display
from sklearn.model_selection import train_test_split

from models.sbi_ariel_adc2023.constants import (
    AUX_FEATURE_COLS as FMPE_AUX_FEATURE_COLS,
    CONTEXT_DIM,
    CONTEXT_FILENAME_TEMPLATE,
    DATASET_TYPE,
    HOLDOUT_SPLIT,
    LOG10_AUX_FEATURE_COLS as FMPE_LOG10_AUX_FEATURE_COLS,
    MANIFEST_FILENAME,
    METADATA_FILENAME_TEMPLATE,
    NORMALIZATION_FILENAME,
    NORMALIZATION_MODE,
    RAW_TARGET_FILENAME_TEMPLATE,
    TARGET_COLS,
    TARGET_FILENAME_TEMPLATE,
    TESTDATA_SPLIT,
    THETA_DIM,
    TRAIN_SPLIT,
    VALIDATION_SPLIT,
    WAVELENGTH_FILENAME,
)
from models.sbi_ariel_adc2023.dataset import load_datasets
from models.sbi_ariel_adc2023.dingo_compat import load_posterior_model
from models.sbi_ariel_adc2023.evaluate import choose_point_estimate, evaluate_split
from models.sbi_ariel_adc2023.training import train_model
from models.ariel_quantum_regression.constants import (
    AUX_COLUMNS as EXOBIOME_AUX_COLUMNS,
    FIXED_SPECTRAL_CHANNELS,
    HDF5_GROUP_PREFIX,
    LOG10_AUX_COLUMNS as EXOBIOME_LOG10_AUX_COLUMNS,
    MODEL_SPECTRAL_CHANNELS,
    PRESENCE_THRESHOLD_LOG10_VMR,
    RAW_SPECTRAL_CHANNELS,
    SAMPLE_SPECTRAL_CHANNELS,
    TARGET_COLUMNS as EXOBIOME_TARGET_COLUMNS,
    WAVELENGTH_DATASET,
)
from models.ariel_quantum_regression.dataset import (
    ArrayStandardizer,
    PreparedData,
    SpectralStandardizer,
    _expected_manifest,
    _make_inference_split,
    _make_labeled_split,
    _normalize_fixed_channel,
    _normalize_sample_spectra,
    _save_prepared_cache,
    transform_aux_features,
)
from models.ariel_quantum_regression.training import TrainingConfig, run_training_experiment

DEVICE = torch.device("cpu")
QUANTUM_DEVICE = "lightning.qubit" if importlib.util.find_spec("pennylane_lightning") is not None else "default.qubit"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cpu_threads = max(1, min(os.cpu_count() or 1, 8))
torch.set_num_threads(cpu_threads)
try:
    torch.set_num_interop_threads(max(1, min(cpu_threads // 2, 4)))
except RuntimeError:
    pass

print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")
print(f"Device: {DEVICE} | Quantum backend: {QUANTUM_DEVICE}")


PyTorch: 2.10.0
PennyLane: 0.44.1
Device: cpu | Quantum backend: lightning.qubit


In [3]:
def _drop_unnamed_columns(frame: pd.DataFrame) -> pd.DataFrame:
    unnamed = [column for column in frame.columns if column.startswith("Unnamed:")]
    if unnamed:
        frame = frame.drop(columns=unnamed)
    return frame


def _load_spectra(hdf5_path: Path, planet_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    spectra = np.empty((len(planet_ids), 52, len(RAW_SPECTRAL_CHANNELS)), dtype=np.float32)
    wavelength_um = None
    with h5py.File(hdf5_path, "r") as handle:
        for row_index, planet_id in enumerate(planet_ids.tolist()):
            group = handle[f"{HDF5_GROUP_PREFIX}{planet_id}"]
            row_wavelength = np.asarray(group[WAVELENGTH_DATASET][:], dtype=np.float32)
            if wavelength_um is None:
                wavelength_um = row_wavelength
            for channel_index, channel_name in enumerate(RAW_SPECTRAL_CHANNELS):
                spectra[row_index, :, channel_index] = np.asarray(group[channel_name][:], dtype=np.float32)
    if wavelength_um is None:
        raise RuntimeError(f"No spectra found in {hdf5_path}")
    return spectra, wavelength_um


train_aux_df = _drop_unnamed_columns(pd.read_csv(DATA_ROOT / "TrainingData" / "AuxillaryTable.csv"))
train_targets_df = _drop_unnamed_columns(
    pd.read_csv(DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv")
)
labels = train_aux_df.merge(train_targets_df[["planet_ID", *TARGET_COLS]], on="planet_ID", how="inner", validate="one_to_one")
labels = labels.reset_index(drop=True)

train_spectra_raw, wavelength_um = _load_spectra(
    DATA_ROOT / "TrainingData" / "SpectralData.hdf5",
    labels["planet_ID"].to_numpy(dtype="U32"),
)
test_aux_df = _drop_unnamed_columns(pd.read_csv(DATA_ROOT / "TestData" / "AuxillaryTable.csv"))
test_spectra_raw, test_wavelength_um = _load_spectra(
    DATA_ROOT / "TestData" / "SpectralData.hdf5",
    test_aux_df["planet_ID"].to_numpy(dtype="U32"),
)
if not np.allclose(wavelength_um, test_wavelength_um, atol=1.0e-8):
    raise AssertionError("Training and test wavelength grids do not match.")

spectrum_index = RAW_SPECTRAL_CHANNELS.index("instrument_spectrum")
noise_index = RAW_SPECTRAL_CHANNELS.index("instrument_noise")

print(f"Training rows: {len(labels):,}")
print(f"Challenge test rows: {len(test_aux_df):,}")
print(f"Wavelength bins: {len(wavelength_um)}")


Training rows: 41,423
Challenge test rows: 685
Wavelength bins: 52


In [4]:
all_indices = np.arange(len(labels), dtype=np.int64)
train_val_pool, holdout_pool = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)
train_pool, validation_pool = train_test_split(
    train_val_pool,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)

rng = np.random.default_rng(SEED)


def take_subset(indices: np.ndarray, limit: int) -> np.ndarray:
    if limit >= len(indices):
        return np.sort(indices.astype(np.int64))
    chosen = rng.choice(indices, size=int(limit), replace=False)
    return np.sort(chosen.astype(np.int64))


train_idx = take_subset(train_pool, TRAIN_ROWS)
validation_idx = take_subset(validation_pool, VALIDATION_ROWS)
holdout_idx = take_subset(holdout_pool, HOLDOUT_ROWS)

split_frame = pd.DataFrame(
    {
        "split": ([TRAIN_SPLIT] * len(train_idx)) + ([VALIDATION_SPLIT] * len(validation_idx)) + ([HOLDOUT_SPLIT] * len(holdout_idx)),
        "planet_ID": np.concatenate(
            [
                labels.iloc[train_idx]["planet_ID"].to_numpy(dtype="U32"),
                labels.iloc[validation_idx]["planet_ID"].to_numpy(dtype="U32"),
                labels.iloc[holdout_idx]["planet_ID"].to_numpy(dtype="U32"),
            ]
        ),
        "source_row_index": np.concatenate([train_idx, validation_idx, holdout_idx]),
    }
)
split_frame.to_csv(OUTPUT_ROOT / "shared_split.csv", index=False)
split_frame.groupby("split").size().to_frame("rows")


,rows
split,
holdout,128
train,512
validation,128


In [5]:
def fit_scaler(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = values.astype(np.float64).mean(axis=0).astype(np.float32)
    scale = values.astype(np.float64).std(axis=0).astype(np.float32)
    scale = np.where(scale == 0.0, 1.0, scale)
    return mean, scale


def safe_scale(values: np.ndarray) -> np.ndarray:
    scale = values.std(axis=0, ddof=0).astype(np.float32)
    return np.where(scale == 0.0, 1.0, scale).astype(np.float32)


def normalize_spectrum(values: np.ndarray) -> np.ndarray:
    sample_mean = values.mean(axis=1, keepdims=True)
    sample_mean = np.clip(sample_mean, 1.0e-12, None)
    return (values / sample_mean).astype(np.float32)


def log_noise(values: np.ndarray) -> np.ndarray:
    return np.log10(np.clip(values.astype(np.float32), 1.0e-12, None)).astype(np.float32)


def compute_zscore(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = values.mean(axis=0).astype(np.float32)
    scale = safe_scale(values)
    return mean, scale


def apply_zscore(values: np.ndarray, mean: np.ndarray, scale: np.ndarray) -> np.ndarray:
    return ((values - mean) / scale).astype(np.float32)


def metadata_frame(indices: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "planet_ID": labels.iloc[indices]["planet_ID"].to_numpy(dtype="U32"),
            "source_row_index": indices.astype(np.int64),
        }
    )


# ExoBiome-style prepared cache from the exact shared split rows.
EXOBIOME_PREPARED_CACHE_DIR = OUTPUT_ROOT / f"exobiome_prepared_shared_t{len(train_idx)}_v{len(validation_idx)}_h{len(holdout_idx)}"
if EXOBIOME_PREPARED_CACHE_DIR.exists():
    shutil.rmtree(EXOBIOME_PREPARED_CACHE_DIR)
EXOBIOME_PREPARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

exobiome_aux_raw = transform_aux_features(labels)
exobiome_test_aux_raw = transform_aux_features(test_aux_df)
exobiome_targets_raw = labels[EXOBIOME_TARGET_COLUMNS].to_numpy(dtype=np.float32, copy=True)
sample_channel_indices = [RAW_SPECTRAL_CHANNELS.index(name) for name in SAMPLE_SPECTRAL_CHANNELS]
width_channel_index = RAW_SPECTRAL_CHANNELS.index("instrument_width")

exobiome_train_sample_spectra = np.transpose(train_spectra_raw[:, :, sample_channel_indices], (0, 2, 1)).astype(np.float32)
exobiome_test_sample_spectra = np.transpose(test_spectra_raw[:, :, sample_channel_indices], (0, 2, 1)).astype(np.float32)
exobiome_train_sample_spectra = _normalize_sample_spectra(exobiome_train_sample_spectra)
exobiome_test_sample_spectra = _normalize_sample_spectra(exobiome_test_sample_spectra)
fixed_channels = np.stack(
    [
        _normalize_fixed_channel(train_spectra_raw[0, :, width_channel_index]),
        _normalize_fixed_channel(wavelength_um),
    ],
    axis=0,
).astype(np.float32)

exobiome_aux_scaler = ArrayStandardizer.fit(exobiome_aux_raw[train_idx])
exobiome_target_scaler = ArrayStandardizer.fit(exobiome_targets_raw[train_idx])
exobiome_spectral_scaler = SpectralStandardizer.fit(exobiome_train_sample_spectra[train_idx], fixed_channels=fixed_channels)


def make_exobiome_labeled_split(indices: np.ndarray):
    aux_scaled = exobiome_aux_scaler.transform(exobiome_aux_raw[indices])
    spectra_scaled = exobiome_spectral_scaler.transform(exobiome_train_sample_spectra[indices])
    raw_targets = exobiome_targets_raw[indices]
    targets_scaled = exobiome_target_scaler.transform(raw_targets)
    return _make_labeled_split(
        planet_ids=labels.iloc[indices]["planet_ID"].to_numpy(dtype="U32"),
        aux_values=aux_scaled,
        spectra_values=spectra_scaled,
        targets_scaled=targets_scaled,
        raw_targets=raw_targets,
    )


exobiome_test_split = _make_inference_split(
    planet_ids=test_aux_df["planet_ID"].to_numpy(dtype="U32"),
    aux_values=exobiome_aux_scaler.transform(exobiome_test_aux_raw),
    spectra_values=exobiome_spectral_scaler.transform(exobiome_test_sample_spectra),
)

exobiome_split_manifest = {
    "rows_total": int(len(labels)),
    "testdata_rows_total": int(len(test_aux_df)),
    "train_rows": int(len(train_idx)),
    "val_rows": int(len(validation_idx)),
    "holdout_rows": int(len(holdout_idx)),
    "testdata_rows": int(len(test_aux_df)),
    "wavelength_bins": int(len(wavelength_um)),
    "wavelength_min_um": float(wavelength_um.min()),
    "wavelength_max_um": float(wavelength_um.max()),
    "raw_spectrum_shape": [52, len(RAW_SPECTRAL_CHANNELS)],
    "model_spectrum_shape": [len(MODEL_SPECTRAL_CHANNELS), 52],
    "aux_columns": EXOBIOME_AUX_COLUMNS,
    "log10_aux_columns": EXOBIOME_LOG10_AUX_COLUMNS,
    "target_columns": EXOBIOME_TARGET_COLUMNS,
    "raw_spectral_channels": RAW_SPECTRAL_CHANNELS,
    "sample_spectral_channels": SAMPLE_SPECTRAL_CHANNELS,
    "fixed_spectral_channels": FIXED_SPECTRAL_CHANNELS,
    "model_spectral_channels": MODEL_SPECTRAL_CHANNELS,
    "sample_spectral_normalization": {
        "mode": "divide_by_sample_mean",
        "reference_channel": SAMPLE_SPECTRAL_CHANNELS[0],
        "applied_channels": SAMPLE_SPECTRAL_CHANNELS,
    },
    "presence_threshold_log10_vmr": PRESENCE_THRESHOLD_LOG10_VMR,
    "split_seed": int(SEED),
    "split_fractions": {"train": 0.8, "val": 0.1, "holdout": 0.1},
    "primary_stratify_mode": "shared_manual_split",
    "secondary_stratify_mode": "shared_manual_split",
    "shared_split_file": str(OUTPUT_ROOT / "shared_split.csv"),
}

exobiome_expected_manifest = _expected_manifest(
    DATA_ROOT,
    SEED,
    int(len(train_idx)),
    int(len(validation_idx)),
    int(len(holdout_idx)),
    None,
)
exobiome_prepared_manifest = dict(exobiome_expected_manifest)
exobiome_prepared_manifest["cache_dir"] = str(EXOBIOME_PREPARED_CACHE_DIR)
exobiome_prepared_manifest["cache_hit"] = False

exobiome_prepared = PreparedData(
    train=make_exobiome_labeled_split(train_idx),
    val=make_exobiome_labeled_split(validation_idx),
    holdout=make_exobiome_labeled_split(holdout_idx),
    testdata=exobiome_test_split,
    aux_scaler=exobiome_aux_scaler,
    target_scaler=exobiome_target_scaler,
    spectral_scaler=exobiome_spectral_scaler,
    wavelength_um=wavelength_um.astype(np.float32),
    split_manifest=exobiome_split_manifest,
    prepared_manifest=exobiome_prepared_manifest,
)
_save_prepared_cache(EXOBIOME_PREPARED_CACHE_DIR, exobiome_expected_manifest, exobiome_prepared)

# FMPE prepared subset from the same labeled rows.
FMPE_PREPARED_DIR = OUTPUT_ROOT / f"fmpe_prepared_shared_t{len(train_idx)}_v{len(validation_idx)}_h{len(holdout_idx)}"
if FMPE_PREPARED_DIR.exists():
    shutil.rmtree(FMPE_PREPARED_DIR)
FMPE_PREPARED_DIR.mkdir(parents=True, exist_ok=True)

fmpe_labeled_spectrum = normalize_spectrum(train_spectra_raw[:, :, spectrum_index])
fmpe_test_spectrum = normalize_spectrum(test_spectra_raw[:, :, spectrum_index])
fmpe_labeled_noise = log_noise(train_spectra_raw[:, :, noise_index])
fmpe_test_noise = log_noise(test_spectra_raw[:, :, noise_index])
fmpe_labeled_aux = labels[FMPE_AUX_FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
fmpe_test_aux = test_aux_df[FMPE_AUX_FEATURE_COLS].to_numpy(dtype=np.float32, copy=True)
for col_index, col_name in enumerate(FMPE_AUX_FEATURE_COLS):
    if col_name in FMPE_LOG10_AUX_FEATURE_COLS:
        fmpe_labeled_aux[:, col_index] = np.log10(np.clip(fmpe_labeled_aux[:, col_index], 1.0e-12, None))
        fmpe_test_aux[:, col_index] = np.log10(np.clip(fmpe_test_aux[:, col_index], 1.0e-12, None))
fmpe_targets_raw = labels[TARGET_COLS].to_numpy(dtype=np.float32, copy=True)

spectrum_mean, spectrum_scale = compute_zscore(fmpe_labeled_spectrum[train_idx])
noise_mean, noise_scale = compute_zscore(fmpe_labeled_noise[train_idx])
aux_mean, aux_scale = compute_zscore(fmpe_labeled_aux[train_idx])
target_mean, target_scale = compute_zscore(fmpe_targets_raw[train_idx])


def fmpe_context(indices: np.ndarray) -> np.ndarray:
    return np.concatenate(
        [
            apply_zscore(fmpe_labeled_spectrum[indices], spectrum_mean, spectrum_scale),
            apply_zscore(fmpe_labeled_noise[indices], noise_mean, noise_scale),
            apply_zscore(fmpe_labeled_aux[indices], aux_mean, aux_scale),
        ],
        axis=1,
    ).astype(np.float32)


def write_labeled_split(split_name: str, indices: np.ndarray) -> None:
    np.save(FMPE_PREPARED_DIR / CONTEXT_FILENAME_TEMPLATE.format(split_name=split_name), fmpe_context(indices))
    np.save(
        FMPE_PREPARED_DIR / TARGET_FILENAME_TEMPLATE.format(split_name=split_name),
        apply_zscore(fmpe_targets_raw[indices], target_mean, target_scale),
    )
    np.save(
        FMPE_PREPARED_DIR / RAW_TARGET_FILENAME_TEMPLATE.format(split_name=split_name),
        fmpe_targets_raw[indices].astype(np.float32),
    )
    metadata_frame(indices).to_csv(
        FMPE_PREPARED_DIR / METADATA_FILENAME_TEMPLATE.format(split_name=split_name),
        index=False,
    )


write_labeled_split(TRAIN_SPLIT, train_idx)
write_labeled_split(VALIDATION_SPLIT, validation_idx)
write_labeled_split(HOLDOUT_SPLIT, holdout_idx)

test_context = np.concatenate(
    [
        apply_zscore(fmpe_test_spectrum, spectrum_mean, spectrum_scale),
        apply_zscore(fmpe_test_noise, noise_mean, noise_scale),
        apply_zscore(fmpe_test_aux, aux_mean, aux_scale),
    ],
    axis=1,
).astype(np.float32)
np.save(FMPE_PREPARED_DIR / CONTEXT_FILENAME_TEMPLATE.format(split_name=TESTDATA_SPLIT), test_context)
pd.DataFrame(
    {
        "planet_ID": test_aux_df["planet_ID"].to_numpy(dtype="U32"),
        "source_row_index": np.arange(len(test_aux_df), dtype=np.int64),
    }
).to_csv(FMPE_PREPARED_DIR / METADATA_FILENAME_TEMPLATE.format(split_name=TESTDATA_SPLIT), index=False)

np.savez(
    FMPE_PREPARED_DIR / NORMALIZATION_FILENAME,
    spectrum_mean=spectrum_mean,
    spectrum_scale=spectrum_scale,
    noise_mean=noise_mean,
    noise_scale=noise_scale,
    aux_mean=aux_mean,
    aux_scale=aux_scale,
    target_mean=target_mean,
    target_scale=target_scale,
)
np.save(FMPE_PREPARED_DIR / WAVELENGTH_FILENAME, wavelength_um.astype(np.float32))

manifest = {
    "data_root": str(DATA_ROOT),
    "prepared_dir": str(FMPE_PREPARED_DIR),
    "dataset_type": DATASET_TYPE,
    "normalization_mode": NORMALIZATION_MODE,
    "seed": SEED,
    "split_fractions": {"train": None, "validation": None, "holdout": None},
    "split_counts": {
        TRAIN_SPLIT: int(len(train_idx)),
        VALIDATION_SPLIT: int(len(validation_idx)),
        HOLDOUT_SPLIT: int(len(holdout_idx)),
        TESTDATA_SPLIT: int(len(test_aux_df)),
    },
    "context_dim": CONTEXT_DIM,
    "theta_dim": THETA_DIM,
    "target_columns": TARGET_COLS,
    "aux_feature_cols": FMPE_AUX_FEATURE_COLS,
    "log10_aux_feature_cols": FMPE_LOG10_AUX_FEATURE_COLS,
    "wavelength_bins": int(len(wavelength_um)),
    "wavelength_min_um": float(wavelength_um.min()),
    "wavelength_max_um": float(wavelength_um.max()),
    "feature_order": [
        "normalized_instrument_spectrum[52]",
        "log10_instrument_noise[52]",
        *FMPE_AUX_FEATURE_COLS,
    ],
    "shared_split_file": str(OUTPUT_ROOT / "shared_split.csv"),
}
(FMPE_PREPARED_DIR / MANIFEST_FILENAME).write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

print(f"ExoBiome prepared cache: {EXOBIOME_PREPARED_CACHE_DIR}")
print(f"FMPE prepared subset: {FMPE_PREPARED_DIR}")


ExoBiome prepared cache: /Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/exobiome_prepared_shared_t512_v128_h128
FMPE prepared subset: /Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/fmpe_prepared_shared_t512_v128_h128


In [6]:
EXOBIOME_OUTPUT_ROOT = OUTPUT_ROOT / "exobiome_small"
if EXOBIOME_OUTPUT_ROOT.exists():
    shutil.rmtree(EXOBIOME_OUTPUT_ROOT)
EXOBIOME_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXOBIOME_STAGE1_OUTPUT_DIR = EXOBIOME_OUTPUT_ROOT / "stage1_classical"
EXOBIOME_STAGE2_OUTPUT_DIR = EXOBIOME_OUTPUT_ROOT / "stage2_hybrid"


def make_exobiome_stage1_config(output_dir: Path) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(EXOBIOME_PREPARED_CACHE_DIR),
        seed=SEED,
        batch_size=EXOBIOME_STAGE1_BATCH_SIZE,
        eval_batch_size=EXOBIOME_STAGE1_EVAL_BATCH_SIZE,
        max_epochs=EXOBIOME_STAGE1_MAX_EPOCHS,
        early_stop_patience=EXOBIOME_STAGE1_EARLY_STOP,
        scheduler_patience=EXOBIOME_STAGE1_SCHEDULER_PATIENCE,
        scheduler_factor=0.5,
        classical_lr=EXOBIOME_STAGE1_CLASSICAL_LR,
        weight_decay=EXOBIOME_WEIGHT_DECAY,
        gradient_clip_norm=EXOBIOME_GRADIENT_CLIP,
        dropout=EXOBIOME_DROPOUT,
        loss_name="mse",
        qnn_qubits=EXOBIOME_QNN_QUBITS,
        qnn_depth=EXOBIOME_QNN_DEPTH,
        qnn_init_scale=EXOBIOME_QNN_INIT_SCALE,
        quantum_device=QUANTUM_DEVICE,
        quantum_use_async=False,
        classical_only=True,
        use_amp=False,
        log_every_batches=10,
        train_limit=len(train_idx),
        val_limit=len(validation_idx),
        holdout_limit=len(holdout_idx),
    )


def make_exobiome_stage2_config(output_dir: Path, checkpoint_path: Path) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(EXOBIOME_PREPARED_CACHE_DIR),
        init_checkpoint_path=str(checkpoint_path),
        seed=SEED,
        batch_size=EXOBIOME_STAGE2_BATCH_SIZE,
        eval_batch_size=EXOBIOME_STAGE2_EVAL_BATCH_SIZE,
        max_epochs=EXOBIOME_STAGE2_MAX_EPOCHS,
        early_stop_patience=EXOBIOME_STAGE2_EARLY_STOP,
        scheduler_patience=EXOBIOME_STAGE2_SCHEDULER_PATIENCE,
        scheduler_factor=0.5,
        classical_lr=EXOBIOME_STAGE2_CLASSICAL_LR,
        quantum_lr=EXOBIOME_STAGE2_QUANTUM_LR,
        weight_decay=EXOBIOME_WEIGHT_DECAY,
        gradient_clip_norm=EXOBIOME_GRADIENT_CLIP,
        dropout=EXOBIOME_DROPOUT,
        loss_name="mse",
        qnn_qubits=EXOBIOME_QNN_QUBITS,
        qnn_depth=EXOBIOME_QNN_DEPTH,
        qnn_init_scale=EXOBIOME_QNN_INIT_SCALE,
        quantum_device=QUANTUM_DEVICE,
        quantum_use_async=False,
        quantum_warmup_epochs=EXOBIOME_STAGE2_WARMUP_EPOCHS,
        quantum_ramp_epochs=EXOBIOME_STAGE2_RAMP_EPOCHS,
        quantum_backbone_freeze_epochs=EXOBIOME_STAGE2_FREEZE_EPOCHS,
        classical_only=False,
        use_amp=False,
        log_every_batches=10,
        train_limit=len(train_idx),
        val_limit=len(validation_idx),
        holdout_limit=len(holdout_idx),
    )


print(
    "ExoBiome two-stage variant | "
    f"qubits={EXOBIOME_QNN_QUBITS}, depth={EXOBIOME_QNN_DEPTH}, "
    f"stage1_epochs={EXOBIOME_STAGE1_MAX_EPOCHS}, stage2_epochs={EXOBIOME_STAGE2_MAX_EPOCHS}"
)


ExoBiome two-stage variant | qubits=12, depth=2, stage1_epochs=18, stage2_epochs=20


In [7]:
exobiome_stage1_cfg = make_exobiome_stage1_config(EXOBIOME_STAGE1_OUTPUT_DIR)
print(json.dumps(exobiome_stage1_cfg.to_json_dict(), indent=2))
exobiome_stage1_result = run_training_experiment(exobiome_stage1_cfg)
print(json.dumps(exobiome_stage1_result["summary"], indent=2))

exobiome_stage1_checkpoint = (EXOBIOME_STAGE1_OUTPUT_DIR / "best_model.pt").resolve()
if not exobiome_stage1_checkpoint.exists():
    raise FileNotFoundError(f"Missing ExoBiome stage-1 checkpoint: {exobiome_stage1_checkpoint}")

exobiome_stage2_cfg = make_exobiome_stage2_config(EXOBIOME_STAGE2_OUTPUT_DIR, exobiome_stage1_checkpoint)
print(json.dumps(exobiome_stage2_cfg.to_json_dict(), indent=2))
exobiome_stage2_result = run_training_experiment(exobiome_stage2_cfg)
print(json.dumps(exobiome_stage2_result["summary"], indent=2))

exobiome_result = {
    "model": "exobiome_two_stage_12q",
    "best_epoch": int(exobiome_stage2_result["summary"]["best_epoch"]),
    "best_val_rmse_mean": float(exobiome_stage2_result["summary"]["best_val_rmse_mean"]),
    "validation_rmse_mean": float(exobiome_stage2_result["summary"]["validation_rmse_mean"]),
    "holdout_rmse_mean": float(exobiome_stage2_result["summary"]["holdout_rmse_mean"]),
    "validation_rmse": {
        name: float(value)
        for name, value in zip(EXOBIOME_TARGET_COLUMNS, exobiome_stage2_result["validation_metrics"]["rmse_orig"])
    },
    "holdout_rmse": {
        name: float(value)
        for name, value in zip(EXOBIOME_TARGET_COLUMNS, exobiome_stage2_result["holdout_metrics"]["rmse_orig"])
    },
    "hyperparameters": {
        "qnn_qubits": EXOBIOME_QNN_QUBITS,
        "qnn_depth": EXOBIOME_QNN_DEPTH,
        "stage1_batch_size": EXOBIOME_STAGE1_BATCH_SIZE,
        "stage1_epochs": EXOBIOME_STAGE1_MAX_EPOCHS,
        "stage1_classical_lr": EXOBIOME_STAGE1_CLASSICAL_LR,
        "stage2_batch_size": EXOBIOME_STAGE2_BATCH_SIZE,
        "stage2_epochs": EXOBIOME_STAGE2_MAX_EPOCHS,
        "stage2_classical_lr": EXOBIOME_STAGE2_CLASSICAL_LR,
        "stage2_quantum_lr": EXOBIOME_STAGE2_QUANTUM_LR,
        "stage2_quantum_ramp_epochs": EXOBIOME_STAGE2_RAMP_EPOCHS,
        "stage2_quantum_backbone_freeze_epochs": EXOBIOME_STAGE2_FREEZE_EPOCHS,
    },
    "stage1_summary": exobiome_stage1_result["summary"],
    "stage2_summary": exobiome_stage2_result["summary"],
}
(EXOBIOME_OUTPUT_ROOT / "summary.json").write_text(json.dumps(exobiome_result, indent=2, sort_keys=True) + "\n")
exobiome_result


{
  "project_root": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks",
  "data_root": "/Users/jkw/Documents/uni/axion/hack4sages/ariel",
  "output_dir": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/exobiome_small/stage1_classical",
  "prepared_cache_dir": "/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/exobiome_prepared_shared_t512_v128_h128",
  "init_checkpoint_path": null,
  "seed": 42,
  "batch_size": 64,
  "eval_batch_size": 128,
  "max_epochs": 18,
  "early_stop_patience": 5,
  "scheduler_patience": 2,
  "scheduler_factor": 0.5,
  "classical_lr": 0.001,
  "quantum_lr": 0.0008,
  "weight_decay": 0.0001,
  "gradient_clip_norm": 5.0,
  "dropout": 0.05,
  "loss_name": "mse",
  "qnn_qubits": 12,
  "qnn_depth": 2,
  "qnn_init_scale": 0.1,
  "quantum_device": "lightning.qubit",
  "quantum_use_async": false,
  "classical_only": true,
  "quantum_warmup_epochs": 5,
  "quantum_ramp_e

{'model': 'exobiome_two_stage_12q',
 'best_epoch': 20,
 'best_val_rmse_mean': 0.8608156442642212,
 'validation_rmse_mean': 0.8608156442642212,
 'holdout_rmse_mean': 0.869857132434845,
 'validation_rmse': {'log_H2O': 0.9702037572860718,
  'log_CO2': 1.1293689012527466,
  'log_CO': 0.8611157536506653,
  'log_CH4': 0.534087061882019,
  'log_NH3': 0.8093026280403137},
 'holdout_rmse': {'log_H2O': 0.9525558948516846,
  'log_CO2': 1.0336004495620728,
  'log_CO': 0.8918188214302063,
  'log_CH4': 0.5932614207267761,
  'log_NH3': 0.8780490756034851},
 'hyperparameters': {'qnn_qubits': 12,
  'qnn_depth': 2,
  'stage1_batch_size': 64,
  'stage1_epochs': 18,
  'stage1_classical_lr': 0.001,
  'stage2_batch_size': 16,
  'stage2_epochs': 20,
  'stage2_classical_lr': 5e-05,
  'stage2_quantum_lr': 0.0002,
  'stage2_quantum_ramp_epochs': 10,
  'stage2_quantum_backbone_freeze_epochs': 4},
 'stage1_summary': {'best_epoch': 17,
  'best_val_rmse_mean': 0.9334365725517273,
  'validation_rmse_mean': 0.9334365

In [8]:
FMPE_OUTPUT_DIR = OUTPUT_ROOT / "fmpe_small"
if FMPE_OUTPUT_DIR.exists():
    shutil.rmtree(FMPE_OUTPUT_DIR)
FMPE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_settings = yaml.safe_load((PROJECT_ROOT / "models" / "sbi_ariel_adc2023" / "settings" / "adc2023_rtx4090.yaml").read_text())
fmpe_settings = json.loads(json.dumps(base_settings))
fmpe_settings["dataset"]["path"] = str(FMPE_PREPARED_DIR)
fmpe_settings["training"]["device"] = "cpu"
fmpe_settings["training"]["batch_size"] = FMPE_BATCH_SIZE
fmpe_settings["training"]["eval_batch_size"] = FMPE_EVAL_BATCH_SIZE
fmpe_settings["training"]["epochs"] = FMPE_MAX_EPOCHS
fmpe_settings["training"]["patience"] = 3
fmpe_settings["training"]["checkpoint_every_batches"] = 5
fmpe_settings["training"]["num_workers"] = 0
fmpe_settings["training"]["pin_memory"] = False
fmpe_settings["training"]["persistent_workers"] = False
fmpe_settings["training"]["prefetch_factor"] = None
fmpe_settings["training"]["max_steps"] = FMPE_MAX_STEPS
fmpe_settings["evaluation"]["device"] = "cpu"
fmpe_settings["evaluation"]["run_after_training"] = False
fmpe_settings["evaluation"]["posterior_samples"] = FMPE_POSTERIOR_SAMPLES
fmpe_settings["evaluation"]["context_batch_size"] = 16
fmpe_settings["evaluation"]["periodic_rmse_every_epochs"] = 1
fmpe_settings["evaluation"]["periodic_posterior_samples"] = FMPE_PERIODIC_SAMPLES
fmpe_settings["evaluation"]["periodic_context_batch_size"] = 16
fmpe_settings["evaluation"]["periodic_rmse_max_rows"] = 0
fmpe_settings["logging"]["use_wandb"] = False
fmpe_settings["task"]["name"] = "adc2023_fivegas_fmpe_small_cpu_compare"

fmpe_summary = train_model(
    settings=fmpe_settings,
    run_dir=FMPE_OUTPUT_DIR,
    prepared_data_override=FMPE_PREPARED_DIR,
    resume_mode="never",
)

fmpe_datasets = load_datasets(fmpe_settings, prepared_data_override=FMPE_PREPARED_DIR)
fmpe_model = load_posterior_model(Path(fmpe_summary["best_model_by_mrmse_path"]), device=DEVICE)
fmpe_validation_metrics, _ = evaluate_split(
    model=fmpe_model,
    dataset=fmpe_datasets["validation"],
    device=DEVICE,
    posterior_samples=FMPE_POSTERIOR_SAMPLES,
    context_batch_size=16,
    include_predictions=False,
)
fmpe_holdout_metrics, _ = evaluate_split(
    model=fmpe_model,
    dataset=fmpe_datasets["holdout"],
    device=DEVICE,
    posterior_samples=FMPE_POSTERIOR_SAMPLES,
    context_batch_size=16,
    include_predictions=False,
)
fmpe_point_estimate = choose_point_estimate("auto", fmpe_validation_metrics)
fmpe_result = {
    "model": "sota_fmpe_small",
    "selected_point_estimate": fmpe_point_estimate,
    "best_epoch": int(fmpe_summary["best_epoch"]),
    "best_val_loss": float(fmpe_summary["best_val_loss"]),
    "validation_rmse_mean": float(fmpe_validation_metrics[fmpe_point_estimate]["rmse_mean"]),
    "holdout_rmse_mean": float(fmpe_holdout_metrics[fmpe_point_estimate]["rmse_mean"]),
    "validation_rmse": {name: float(value) for name, value in fmpe_validation_metrics[fmpe_point_estimate]["rmse"].items()},
    "holdout_rmse": {name: float(value) for name, value in fmpe_holdout_metrics[fmpe_point_estimate]["rmse"].items()},
    "training_summary": fmpe_summary,
}
(FMPE_OUTPUT_DIR / "summary.json").write_text(json.dumps(fmpe_result, indent=2, sort_keys=True) + "\n")
fmpe_result


Putting posterior model to device cpu.
Start training epoch 1 on cpu.
Epoch 1 complete | train_loss=1.947205 val_loss=1.892598 lr=4.999452e-04 best_val_loss=inf best_rmse=n/a
Epoch 1 RMSE monitor | device=cpu rows=128/128 posterior_samples=16
Epoch 1 RMSE | mean=1.392971 median=1.399230 rows=128 samples=16 device=cpu
Start training epoch 2 on cpu.
Epoch 2 complete | train_loss=1.623195 val_loss=1.660619 lr=4.997807e-04 best_val_loss=1.892598 best_rmse=1.392971
Epoch 2 RMSE monitor | device=cpu rows=128/128 posterior_samples=16
Epoch 2 RMSE | mean=1.275771 median=1.301796 rows=128 samples=16 device=cpu
Start training epoch 3 on cpu.
Epoch 3 complete | train_loss=1.597078 val_loss=1.850569 lr=4.995067e-04 best_val_loss=1.660619 best_rmse=1.275771
Epoch 3 RMSE monitor | device=cpu rows=128/128 posterior_samples=16
Epoch 3 RMSE | mean=1.254896 median=1.266075 rows=128 samples=16 device=cpu
Start training epoch 4 on cpu.
Epoch 4 complete | train_loss=1.363074 val_loss=1.395653 lr=4.991232e-

{'model': 'sota_fmpe_small',
 'selected_point_estimate': 'mean',
 'best_epoch': 6,
 'best_val_loss': 1.3092398643493652,
 'validation_rmse_mean': 0.9729164361953735,
 'holdout_rmse_mean': 1.1225930094718932,
 'validation_rmse': {'log_H2O': 1.0728150606155396,
  'log_CO2': 1.1673237085342407,
  'log_CO': 0.8504146337509155,
  'log_CH4': 0.8661669492721558,
  'log_NH3': 0.9078618288040161},
 'holdout_rmse': {'log_H2O': 1.371421217918396,
  'log_CO2': 1.2183970212936401,
  'log_CO': 0.8626510500907898,
  'log_CH4': 1.0458704233169556,
  'log_NH3': 1.114625334739685},
 'training_summary': {'status': 'completed',
  'run_dir': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/fmpe_small',
  'resume_checkpoint': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/fmpe_small/resume_latest.pt',
  'best_model_path': '/Users/jkw/Documents/uni/axion/hack4sages/ariel/models/notebooks/outputs/small_data_compare/fmpe_

In [9]:
comparison_df = pd.DataFrame(
    [
        {
            "model": exobiome_result["model"],
            "validation_rmse_mean": exobiome_result["validation_rmse_mean"],
            "holdout_rmse_mean": exobiome_result["holdout_rmse_mean"],
            "notes": (
                f"two_stage, qubits={EXOBIOME_QNN_QUBITS}, "
                f"stage1_epochs={EXOBIOME_STAGE1_MAX_EPOCHS}, stage2_epochs={EXOBIOME_STAGE2_MAX_EPOCHS}"
            ),
        },
        {
            "model": fmpe_result["model"],
            "validation_rmse_mean": fmpe_result["validation_rmse_mean"],
            "holdout_rmse_mean": fmpe_result["holdout_rmse_mean"],
            "notes": f"point_estimate={fmpe_result['selected_point_estimate']}, epochs={FMPE_MAX_EPOCHS}, max_steps={FMPE_MAX_STEPS}",
        },
    ]
).sort_values("holdout_rmse_mean", ascending=True, ignore_index=True)

winner = comparison_df.iloc[0]
summary = {
    "winner": winner["model"],
    "comparison": comparison_df.to_dict(orient="records"),
    "shared_split_file": str(OUTPUT_ROOT / "shared_split.csv"),
    "exobiome_prepared_cache_dir": str(EXOBIOME_PREPARED_CACHE_DIR),
    "fmpe_prepared_dir": str(FMPE_PREPARED_DIR),
}
(OUTPUT_ROOT / "comparison_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")

display(comparison_df)
print(f"Winner on holdout mean RMSE: {winner['model']}")


,model,validation_rmse_mean,holdout_rmse_mean,notes
0,exobiome_two_stage_12q,0.860816,0.869857,"two_stage, qubits=12, stage1_epochs=18, stage2..."
1,sota_fmpe_small,0.972916,1.122593,"point_estimate=mean, epochs=6, max_steps=60"


Winner on holdout mean RMSE: exobiome_two_stage_12q
